In [8]:
def normalize(text: str) -> str:
    """规范化字符串：转为小写、去除空格和连字符（与 langref.py 的 judge_correctness 一致）"""
    return text.lower().strip().replace("-", "").replace(" ", "")

In [66]:
# 统计生成结果中，原product字段为空的样本，生成结果也为空的数量

# 导入per8_model.jsonl预测结果
import jsonlines
pred_results = []
with jsonlines.open('../OUTPUTs/per0.5_model.jsonl', 'r') as reader:
    for item in reader:
        pred_results.append(item)

empty_count = 0
for item in pred_results:
    original_product = item.get('product', '').strip()
    predicted_product = item.get('extracted_models', [])[0] if item.get('extracted_models') else ''
    if original_product == '' and predicted_product == '':
        empty_count += 1
print(f"原product字段为空，生成结果也为空的样本数量: {empty_count}")
# 负样本过滤率 = 原product字段为空，生成结果也为空的样本数量 / 原product字段为空的样本总数量
total_empty_original = sum(1 for item in pred_results if item.get('product', '').strip() == '')
if total_empty_original > 0:
    filtering_rate = empty_count / total_empty_original
    print(f"负样本过滤率: {filtering_rate:.2%}")

原product字段为空，生成结果也为空的样本数量: 4831
负样本过滤率: 98.89%


In [67]:
# 统计生成结果中，原product字段不为空的样本，生成结果也不为空的数量

non_empty_count = 0
for item in pred_results:
    original_product = item.get('product', '').strip() 
    predicted_product = item.get('extracted_models', [])[0] if item.get('extracted_models') else ''
    if original_product != '' and predicted_product != '':
        non_empty_count += 1
print(f"原product字段不为空，生成结果也不为空的样本数量: {non_empty_count}")
# 正样本保留率 = 原product字段不为空，生成结果也不为空的样本数量 / 原product字段不为空的样本总数量
total_non_empty_original = sum(1 for item in pred_results if item.get('product', '').strip() != '')
if total_non_empty_original > 0:
    retention_rate = non_empty_count / total_non_empty_original
    print(f"正样本保留率: {retention_rate:.2%}")

原product字段不为空，生成结果也不为空的样本数量: 8192
正样本保留率: 91.48%


In [46]:
# 统计生成结果中，原product字段不为空的样本，生成结果为空的数量

empty_from_non_empty_count = 0
for item in pred_results:
    original_product = item.get('product', ' ').strip()
    predicted_product = item.get('extracted_models', [])[0] if item.get('extracted_models') else ''
    if original_product != '' and predicted_product == '':
        empty_from_non_empty_count += 1
print(f"原product字段不为空，生成结果为空的样本数量: {empty_from_non_empty_count}")
# 正样本误删率 = 原product字段不为空，生成结果为空的样本数量 / 原product字段不为空的样本总数量    
if total_non_empty_original > 0:
    false_negative_rate = empty_from_non_empty_count / total_non_empty_original
    print(f"正样本误删率: {false_negative_rate:.2%}")

原product字段不为空，生成结果为空的样本数量: 435
正样本误删率: 4.86%


# 小模型结果的分类

In [54]:
# 导入per1_model.jsonl预测结果
import jsonlines
pred_results = []
with jsonlines.open('../OUTPUTs/per1_model.jsonl', 'r') as reader:
    for item in reader:
        pred_results.append(item)

# 有小模型提取结果的为有效样本
valid_samples = [item for item in pred_results if item.get('extracted_models')]
print(f"有效样本数量: {len(valid_samples)}")

有效样本数量: 8615


In [57]:
"""
重新定义样本类型：
1. 样本类型TT：原product字段和生成结果都不为空的样本
2. 样本类型NT：原product字段为空，生成结果不为空的样本（小模型错判）
3. 样本类型TN：原product字段不为空，生成结果为空的样本（小模型漏判）
4. 样本类型NN：原product字段和生成结果都为空的样本
"""
# 将pred_results中的sample_type字段修改
for item in pred_results:
    original_product = item.get('product', '').strip()
    predicted_product = item.get('extracted_models', [])[0] if item.get('extracted_models') else ''
    
    if original_product != '' and predicted_product != '':
        item['sample_type'] = 'TT'
    elif original_product == '' and predicted_product != '':
        item['sample_type'] = 'NT'
    elif original_product != '' and predicted_product == '':
        item['sample_type'] = 'TN'
    elif original_product == '' and predicted_product == '':
        item['sample_type'] = 'NN'

In [55]:
# 将原product字段为空，生成结果不为空的样本筛选出来
slm_wrong_samples = []
for item in valid_samples:
    original_product = item.get('product', '').strip()
    predicted_product = item.get('extracted_models', [])[0] if item.get('extracted_models') else ''
    if original_product == '' and predicted_product != '':
        slm_wrong_samples.append(item)

print(f"原product字段为空，生成结果不为空的样本数量: {len(slm_wrong_samples)}")

原product字段为空，生成结果不为空的样本数量: 95


In [60]:
# 有小模型提取结果的为有效样本
valid_samples = [item for item in pred_results if item.get('extracted_models')]
print(f"有效样本数量: {len(valid_samples)}")

有效样本数量: 8615


In [63]:
from collections import Counter
# 输出pred_results的sample_type统计
sample_type_counter = Counter()
for item in pred_results:
    sample_type = item.get('sample_type', 'Unknown')
    sample_type_counter[sample_type] += 1
print("样本类型统计:")
for sample_type, count in sample_type_counter.items():
    print(f"  {sample_type}: {count}")

样本类型统计:
  TT: 8520
  TN: 435
  NN: 4790
  NT: 95


## 过滤后样本统计

In [626]:
# 读取final_llm_testset.jsonl文件
import jsonlines

testset = []
with jsonlines.open('../OUTPUTs/final_llm_testset.jsonl') as reader:
    for obj in reader:
        testset.append(obj)

# 查看testset的设备类型分布
from collections import Counter

device_types = [item.get('type') for item in testset]
device_type_counter = Counter(device_types)

# 总共有多少种设备类型
print(f"总共有 {len(device_type_counter)} 种设备类型。")

# 统计主流类型（前11名）样本量和占比
mainstream_types = device_type_counter.most_common(11)
total_count = sum(device_type_counter.values())
for device_type, count in mainstream_types:
    proportion = count / total_count * 100
    print(f"  {device_type}: {count} ({proportion:.2f}%)")

# 前1名类型占总样本量的比例
mainstream_total = sum(count for _, count in mainstream_types)
mainstream_proportion = mainstream_total / total_count * 100
print(f"前11名类型占总样本量的比例: {mainstream_proportion:.2f}%")

总共有 77 种设备类型。
  NVR: 1833 (21.28%)
  DVR: 1592 (18.48%)
  IPCam: 1468 (17.04%)
  NAS: 1210 (14.05%)
  Printer: 602 (6.99%)
  HLR: 588 (6.83%)
  Router: 245 (2.84%)
  Firewall_H: 172 (2.00%)
  Switch: 126 (1.46%)
  : 95 (1.10%)
  IAD: 88 (1.02%)
前11名类型占总样本量的比例: 93.08%


In [629]:
# 统计final_per1_pred.jsonl中的设备类型分布
import jsonlines

testset = []
with jsonlines.open('../OUTPUTs/final_per1_pred.jsonl') as reader:
    for obj in reader:
        testset.append(obj)

# 查看testset的设备类型分布
from collections import Counter

device_types = [item.get('type') for item in testset]
device_type_counter = Counter(device_types)

# 总共有多少种设备类型
print(f"总共有 {len(device_type_counter)} 种设备类型。")

# 统计所有类型样本量和占比，倒叙输出
total_count = sum(device_type_counter.values())
for device_type, count in device_type_counter.most_common():
    proportion = count / total_count * 100
    print(f"  {device_type}: {count} ({proportion:.2f}%)")

总共有 80 种设备类型。
  : 4885 (35.30%)
  NVR: 1841 (13.30%)
  DVR: 1609 (11.63%)
  IPCam: 1513 (10.93%)
  NAS: 1216 (8.79%)
  Printer: 679 (4.91%)
  HLR: 608 (4.39%)
  Router: 277 (2.00%)
  Firewall_H: 176 (1.27%)
  Switch: 145 (1.05%)
  IPTel: 120 (0.87%)
  IAD: 88 (0.64%)
  PLC: 75 (0.54%)
  IEG: 52 (0.38%)
  Gateway: 47 (0.34%)
  OpticalTransEquip: 40 (0.29%)
  IRouter: 33 (0.24%)
  WAP: 31 (0.22%)
  Bridge: 30 (0.22%)
  SDS: 23 (0.17%)
  PVInverter: 19 (0.14%)
  VCT: 19 (0.14%)
  Modem: 18 (0.13%)
  UTM: 16 (0.12%)
  AVPowerAmp: 15 (0.11%)
  BASC: 15 (0.11%)
  IPPBX: 15 (0.11%)
  SmartTV: 15 (0.11%)
  WRE: 14 (0.10%)
  WireReceiver: 13 (0.09%)
  SOC: 12 (0.09%)
  VGW: 10 (0.07%)
  DVS: 10 (0.07%)
  HMI: 9 (0.07%)
  VMS_H: 9 (0.07%)
  OthRSS: 8 (0.06%)
  ICSCellGW: 8 (0.06%)
  NVS: 8 (0.06%)
  RTU: 8 (0.06%)
  WAPC: 8 (0.06%)
  VideoCodec: 7 (0.05%)
  ServDev: 7 (0.05%)
  MCU: 6 (0.04%)
  DAM: 6 (0.04%)
  VoIPRouter: 5 (0.04%)
  SAS_H: 5 (0.04%)
  MSG_H: 5 (0.04%)
  CPIE: 4 (0.03%)
  ICSGW

# SLM错误结果分析

In [39]:
# 将final_llm_testset.jsonl中product和SLM结果不一致的样本筛选出来
# 字符串对比的方式参考slm_evalue.py
import jsonlines
import json

def normalize(text: str) -> str:
    """规范化字符串：转为小写、去除空格和连字符（与 slm_evalue.py 一致）"""
    return text.lower().strip().replace("-", "").replace(" ", "")

# 将final_llm_testset.jsonl中product和SLM结果不一致的样本筛选出来
# 字符串对比的方式参考slm_evalue.py
mismatched_samples = []

with jsonlines.open('../OUTPUTs/final_llm_testset.jsonl') as reader:
    for obj in reader:
        product = obj.get("product", "")
        extracted_models = obj.get("extracted_models", [])
        
        # 规范化product
        n_product = normalize(product)
        
        # 检查是否有匹配的SLM结果
        matched = False
        for ext in extracted_models:
            if isinstance(ext, list) and len(ext) > 0:
                s = str(ext[0]).strip()  # 取第一个元素作为型号字符串
            else:
                s = str(ext).strip()
            if normalize(s) == n_product:
                matched = True
                break
        
        # 如果没有匹配，则为不一致样本
        if not matched:
            mismatched_samples.append(obj)

print(f"product和SLM结果不一致的样本数量: {len(mismatched_samples)}")

# 保存这些样本到jsonl文件
with jsonlines.open('mismatched_samples.jsonl', mode='w') as writer:
    writer.write_all(mismatched_samples)

print("不一致样本已保存到 mismatched_samples.jsonl")

product和SLM结果不一致的样本数量: 696
不一致样本已保存到 mismatched_samples.jsonl


In [616]:
# 比较SLM和LLM结果准确率，正确值为product字段
# 用同样的规范化函数（分别与product比较），SLM结果取extracted_models的第一个元素，LLM结果取llm_result字段
# 最终输出SLM和LLM的准确率
import jsonlines

def normalize(text: str) -> str:
    """规范化字符串：转为小写、去除空格和连字符"""
    if text is None:
        return ""
    return text.lower().strip().replace("-", "").replace(" ", "")

# 读取sllm_combine.jsonl
slm_correct = 0
llm_correct = 0
total_samples = 0

COM_FILE = '../OUTPUTs/sllm_combine.jsonl'
# COM_FILE = '../OUTPUTs/tn_nt_res.jsonl'

with jsonlines.open(COM_FILE) as reader:
    for obj in reader:
        product = obj.get("product", "")
        extracted_models = obj.get("extracted_models", "models")
        llm_result = obj.get("llm_result", "")
        
        n_product = normalize(product)
        
        # 提取SLM模型列表
        slm_models = []
        if isinstance(extracted_models, list):
            for ext in extracted_models:
                if isinstance(ext, list) and len(ext) > 0:
                    slm_models.append(str(ext[0]).strip())
                elif isinstance(ext, str):
                    slm_models.append(ext.strip())
        elif isinstance(extracted_models, dict):
            slm_models.append(extracted_models.get("model", ""))
        
        # 检查SLM结果
        slm_matched = any(normalize(s) == n_product for s in slm_models)
        
        # 检查LLM结果
        llm_matched = normalize(llm_result) == n_product
        
        if slm_matched:
            slm_correct += 1
        if llm_matched:
            llm_correct += 1
        
        total_samples += 1

# 输出准确率
slm_accuracy = slm_correct / total_samples if total_samples > 0 else 0
llm_accuracy = llm_correct / total_samples if total_samples > 0 else 0

print(f"SLM准确率: {slm_accuracy:.2%}")
print(f"LLM准确率: {llm_accuracy:.2%}")

SLM准确率: 91.08%
LLM准确率: 91.54%


In [617]:
     # 将final_llm_testset.jsonl中product和LLM结果不一致的样本筛选出来
# 代码参考slm的
import jsonlines

def filter_inconsistent_samples(input_file: str, output_file: str):
    with jsonlines.open(input_file) as reader, jsonlines.open(output_file, mode='w') as writer:
        num_mismatched = 0
        for obj in reader:
            product = obj.get("product", "")
            llm_result = obj.get("llm_result", "")
            if normalize(llm_result) != normalize(product):
                writer.write(obj)
                num_mismatched += 1
        print(f"product和LLM结果不一致的样本数量: {num_mismatched}")

filter_inconsistent_samples('../OUTPUTs/sllm_combine.jsonl', 'llm_mismatched_samples.jsonl')
print("LLM不一致样本已保存到 llm_mismatched_samples.jsonl")

product和LLM结果不一致的样本数量: 687
LLM不一致样本已保存到 llm_mismatched_samples.jsonl


In [601]:
# 评估LLM结果的准确率，参考slm_evalue.py的计算方式
import jsonlines
import json
import os

def normalize(text: str) -> str:
    """规范化字符串：转为小写、去除空格和连字符（与 langref.py 的 judge_correctness 一致）"""
    return text.lower().strip().replace("-", "").replace(" ", "")

def evaluate_llm_extractions(result_path="OUTPUTs/sllm_combine.jsonl"):
    """
    评估LLM提取结果的准确率。
    类似slm_evalue.py，但针对llm_result字段。
    """
    total_samples = 0
    positive_samples = 0  # product非空样本
    recall_hit_samples = 0  # LLM结果与product匹配的样本数（仅对正样本）
    correct_count = 0  # 匹配的样本数
    extracted_empty_count = 0  # llm_result为空的样本数
    extracted_nonempty_count = 0  # llm_result非空的样本数

    with jsonlines.open(result_path, "r") as f:
        for item in f:
            total_samples += 1
            product = str(item.get("product", "")).strip()
            llm_result = str(item.get("llm_result", "")).strip()

            if not llm_result:
                extracted_empty_count += 1
            else:
                extracted_nonempty_count += 1

            n_product = normalize(product)
            n_llm = normalize(llm_result)

            if product:  # 正样本
                positive_samples += 1
                if n_llm == n_product:
                    recall_hit_samples += 1
                    correct_count += 1
            else:  # 负样本
                if n_llm == n_product:  # 空匹配空
                    correct_count += 1

    # 计算指标
    recall = (recall_hit_samples / positive_samples) if positive_samples > 0 else 0.0
    accuracy = (correct_count / total_samples) if total_samples > 0 else 0.0

    metrics = {
        "total_samples": total_samples,
        "positive_samples": positive_samples,
        "recall_hit_samples": recall_hit_samples,
        "correct_count": correct_count,
        "extracted_empty_count": extracted_empty_count,
        "extracted_nonempty_count": extracted_nonempty_count,
        "recall": recall,
        "accuracy": accuracy
    }

    print("LLM评估结果:")
    print(f"总样本数: {total_samples}")
    print(f"正样本数: {positive_samples}")
    print(f"召回命中样本数: {recall_hit_samples}")
    print(f"正确样本数: {correct_count}")
    print(f"LLM结果为空样本数: {extracted_empty_count}")
    print(f"LLM结果非空样本数: {extracted_nonempty_count}")
    print(f"召回率: {recall:.4f}")
    print(f"准确率: {accuracy:.4f}")

    return metrics 

# 执行评估
result_path = "../OUTPUTs/sllm_combine.jsonl"
if os.path.exists(result_path):
    evaluate_llm_extractions(result_path)
else:
    print(f"文件不存在: {result_path}")

LLM评估结果:
总样本数: 8615
正样本数: 8520
召回命中样本数: 7746
正确样本数: 7747
LLM结果为空样本数: 15
LLM结果非空样本数: 8600
召回率: 0.9092
准确率: 0.8992


In [611]:
# 将final_per1_pred.jsonl中NT和TN类型样本筛选出来
import jsonlines

def filter_tn_samples(input_path: str, output_path: str):
    with jsonlines.open(input_path) as reader, jsonlines.open(output_path, mode='w') as writer:
        for obj in reader:
            if obj.get("sample_type") == "TN" or obj.get("sample_type") == "NT":
                writer.write(obj)

filter_tn_samples('../OUTPUTs/final_per1_pred.jsonl', '../OUTPUTs/tn_nt_samples.jsonl')
print("TN和NT类型样本已保存到 tn_nt_samples.jsonl")

TN和NT类型样本已保存到 tn_nt_samples.jsonl


In [618]:
# 读取llm_mismatched_samples.jsonl文件，仅提取product和llm_result字段，保存为ai可以读取的文件类型
import jsonlines

with jsonlines.open('llm_mismatched_samples.jsonl') as reader:
    with jsonlines.open('llm_mismatched_samples_filtered.jsonl', mode='w') as writer:
        for obj in reader:
            filtered = {
                'product': obj.get('product'),
                'llm_result': obj.get('llm_result')
            }
            writer.write(filtered)

In [619]:
# 把llm_mismatched_samples_filtered.jsonl保存为txt文件
import jsonlines

with jsonlines.open('llm_mismatched_samples_filtered.jsonl') as reader:
    with open('llm_mismatched_samples_filtered.txt', mode='w', encoding='utf-8') as writer:
        for obj in reader:
            writer.write(f"Product: {obj.get('product')}\n")
            writer.write(f"LLM Result: {obj.get('llm_result')}\n")
            writer.write("\n")

In [631]:
# 统计sllm_combine.jsonl中call_agents的分布
import jsonlines
from collections import Counter

call_agents = []
with jsonlines.open('../OUTPUTs/sllm_combine.jsonl') as reader:
    for obj in reader:
        ca = obj.get('call_agents', '')
        # 支持三种常见存储形式：字符串 "001"，列表 ['0','0','1']，或列表 ['001']
        if isinstance(ca, list):
            # 如果是单字符列表，合并为完整字符串
            if all(isinstance(x, str) and len(x) == 1 for x in ca):
                call_agents.append(''.join(ca))
            else:
                # 否则把列表元素合并为逗号分隔的字符串，保留原意
                call_agents.append(','.join(map(str, ca)))
        else:
            call_agents.append(str(ca))

# 统计call_agents的分布（以完整条目为单位）
call_agent_counter = Counter(call_agents)

# 总共有多少种call_agent
print(f"总共有 {len(call_agent_counter)} 种 call_agent。")

# 统计所有类型样本量和占比，倒叙输出
total_count = sum(call_agent_counter.values())
for call_agent, count in call_agent_counter.most_common():
    proportion = count / total_count * 100 if total_count else 0
    print(f"  {call_agent}: {count} ({proportion:.2f}%)")

总共有 6 种 call_agent。
  000: 7998 (98.53%)
  001: 53 (0.65%)
  010: 23 (0.28%)
  100: 19 (0.23%)
  101: 12 (0.15%)
  011: 12 (0.15%)


In [637]:
# 判断所有call_agents不是'000'的样本的准确率
# 准确：llm_result = product（规范化后相等）
import jsonlines

llm_correct = 0  # 初始化变量
slm_correct = 0  # 初始化变量
total = 0
with jsonlines.open('../OUTPUTs/sllm_combine.jsonl') as reader:
    for obj in reader:
        call_agents = obj.get('call_agents', '')
        if '000' not in str(call_agents):  # 确保call_agents是字符串
            product = obj.get('product', '')
            llm_result = obj.get('llm_result', '')
            slm_result = obj.get('extracted_models', {})  # extracted_models是字典
            if normalize(product) == normalize(llm_result):
                llm_correct += 1
            # 正确访问SLM模型：从字典中获取"model"字段
            slm_model = slm_result.get("model", "") if isinstance(slm_result, dict) else ""
            if normalize(product) == normalize(slm_model):
                slm_correct += 1
            total += 1
llm_accuracy = llm_correct / total if total > 0 else 0
slm_accuracy = slm_correct / total if total > 0 else 0

print(f"所有call_agents不是'000'的LLM准确率: {llm_accuracy:.2%}")
print(f"所有call_agents不是'000'的SLM准确率: {slm_accuracy:.2%}")

所有call_agents不是'000'的LLM准确率: 51.26%
所有call_agents不是'000'的SLM准确率: 20.17%


In [641]:
# 所有call_agents是'001'or'011'or'101'的LLM准确率
import jsonlines

llm_correct = 0  # 初始化变量
slm_correct = 0  # 初始化变量
total = 0
with jsonlines.open('../OUTPUTs/sllm_combine.jsonl') as reader:
    for obj in reader:
        call_agents = obj.get('call_agents', '')
        if '001' in str(call_agents) or '011' in str(call_agents) or '101' in str(call_agents):  # 确保call_agents是字符串
            product = obj.get('product', '')
            llm_result = obj.get('llm_result', '')
            slm_result = obj.get('extracted_models', {})  # extracted_models是字典
            if normalize(product) == normalize(llm_result):
                llm_correct += 1
            # 正确访问SLM模型：从字典中获取"model"字段
            slm_model = slm_result.get("model", "") if isinstance(slm_result, dict) else ""
            if normalize(product) == normalize(slm_model):
                slm_correct += 1
            total += 1
llm_accuracy = llm_correct / total if total > 0 else 0
slm_accuracy = slm_correct / total if total > 0 else 0

print(f"所有call_agents是'001'或'011'或'101'的LLM准确率: {llm_accuracy:.2%}")
print(f"所有call_agents是'001'或'011'或'101'的SLM准确率: {slm_accuracy:.2%}")

所有call_agents是'001'或'011'或'101'的LLM准确率: 54.55%
所有call_agents是'001'或'011'或'101'的SLM准确率: 14.29%


In [643]:
# 所有call_agents是'010'的LLM准确率--Searcher
import jsonlines

llm_correct = 0  # 初始化变量
slm_correct = 0  # 初始化变量
total = 0
with jsonlines.open('../OUTPUTs/sllm_combine.jsonl') as reader:
    for obj in reader:
        call_agents = obj.get('call_agents', '')
        if '010' in str(call_agents):  # 确保call_agents是字符串
            product = obj.get('product', '')
            llm_result = obj.get('llm_result', '')
            slm_result = obj.get('extracted_models', {})  # extracted_models是字典
            if normalize(product) == normalize(llm_result):
                llm_correct += 1
            # 正确访问SLM模型：从字典中获取"model"字段
            slm_model = slm_result.get("model", "") if isinstance(slm_result, dict) else ""
            if normalize(product) == normalize(slm_model):
                slm_correct += 1
            total += 1
llm_accuracy = llm_correct / total if total > 0 else 0
slm_accuracy = slm_correct / total if total > 0 else 0

print(f"所有call_agents是'010'的LLM准确率: {llm_accuracy:.2%}")
print(f"所有call_agents是'010'的SLM准确率: {slm_accuracy:.2%}")

所有call_agents是'010'的LLM准确率: 56.52%
所有call_agents是'010'的SLM准确率: 52.17%


In [640]:
# 所有call_agents是'100'的LLM准确率
import jsonlines
llm_correct = 0  # 初始化变量
slm_correct = 0  # 初始化变量
total = 0
with jsonlines.open('../OUTPUTs/sllm_combine.jsonl') as reader:
    for obj in reader:
        call_agents = obj.get('call_agents', '')
        if '100' in str(call_agents):  # 确保call_agents是字符串
            product = obj.get('product', '')
            llm_result = obj.get('llm_result', '')
            slm_result = obj.get('extracted_models', {})  # extracted_models是字典
            if normalize(product) == normalize(llm_result):
                llm_correct += 1
            # 正确访问SLM模型：从字典中获取"model"字段
            slm_model = slm_result.get("model", "") if isinstance(slm_result, dict) else ""
            if normalize(product) == normalize(slm_model):
                slm_correct += 1
            total += 1
llm_accuracy = llm_correct / total if total > 0 else 0
slm_accuracy = slm_correct / total if total > 0 else 0

print(f"所有call_agents是'100'的LLM准确率: {llm_accuracy:.2%}")
print(f"所有call_agents是'100'的SLM准确率: {slm_accuracy:.2%}")

所有call_agents是'100'的LLM准确率: 31.58%
所有call_agents是'100'的SLM准确率: 5.26%
